# COMPARATIVA DE ABLACIONES

Este cuaderno sirve para analizar los resultados generados en `data/results/modeling/ablation`.


### 1. PREPARACIÓN.
Actualiza la ruta `ABLATION_DIR`.

In [1]:
from pathlib import Path
import pandas as pd
import plotly.express as px

ABLATION_DIR = Path('../data/results/modeling/ablation')
ABLATION_DIR

PosixPath('../data/results/modeling/ablation')

In [2]:
summaries = []
for subdir in ABLATION_DIR.iterdir():
    if not subdir.is_dir():
        continue
    for scenario_dir in subdir.iterdir():
        if not scenario_dir.is_dir():
            continue
        summary_path = scenario_dir / 'summary.csv'
        if summary_path.exists():
            df = pd.read_csv(summary_path)
            df['scenario'] = scenario_dir.name
            df['ablation'] = subdir.name
            summaries.append(df)

if not summaries:
    raise FileNotFoundError("No summary.csv files found under ablation directory.")

ablation_df = pd.concat(summaries, ignore_index=True)
ablation_df.head()


,model,split,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std,med_ae_mean,med_ae_std,max_err_mean,max_err_std,samples_total,scenario,ablation
0,catboost,cv,0.135780,0.014734,0.171279,0.007928,-0.032980,0.082898,0.105799,0.024019,0.453395,0.043047,12451,runner_id_20260103_205321,no_accelerations_jerk_velocity
1,catboost,test,0.154611,NaN,0.179700,NaN,0.045245,NaN,0.133832,NaN,0.374939,NaN,2942,runner_id_20260103_205321,no_accelerations_jerk_velocity
2,elasticnet,cv,0.139073,0.017491,0.172761,0.012358,-0.048332,0.088594,0.110564,0.026723,0.434598,0.051821,12451,runner_id_20260103_205321,no_accelerations_jerk_velocity
3,elasticnet,test,0.160286,NaN,0.182120,NaN,0.019351,NaN,0.150399,NaN,0.420753,NaN,2942,runner_id_20260103_205321,no_accelerations_jerk_velocity
4,gradient_boosting,cv,0.136415,0.014203,0.170998,0.007983,-0.029518,0.081800,0.108373,0.023024,0.470973,0.037635,12451,runner_id_20260103_205321,no_accelerations_jerk_velocity


In [3]:
metrics_table = (
    ablation_df
    .pivot_table(index=['model'], columns='scenario', values='r2_mean')
    .round(3)
)
metrics_table

scenario,runner_id_20260103_191454,runner_id_20260103_205321,runner_id_20260103_213002
model,,,
catboost,0.760,0.006,0.718
elasticnet,0.664,-0.014,0.657
gradient_boosting,0.763,0.005,0.732
hist_gradient_boosting,0.751,-0.016,0.717
random_forest,0.742,0.011,0.725
xgboost,0.759,-0.020,0.711


### 2. COMPARATIVA DE MÉTRICAS.
Gráficas para comparar MAE/RMSE/R² por modelo y ablación.

In [4]:
fig = px.bar(
    ablation_df[ablation_df['split']=='test'],
    x='model',
    y='mae_mean',
    color='ablation',
    barmode='group',
    title='Comparación de MAE por experimento de ablación (test)',
    text_auto='.4f'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='MAE medio (test)')
fig.show()

In [5]:
fig = px.bar(
    ablation_df[ablation_df['split']=='test'],
    x='model',
    y='rmse_mean',
    color='ablation',
    barmode='group',
    title='Comparación de RMSE por experimento de ablación (test)',
    text_auto='.4f'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='RMSE medio (test)')
fig.show()


In [6]:
fig = px.bar(
    ablation_df[ablation_df['split']=='test'],
    x='model',
    y='r2_mean',
    color='ablation',
    barmode='group',
    title='Comparación de R² por experimento de ablación (test)',
    text_auto='.4f'
)
fig.update_layout(xaxis_title='Modelo', yaxis_title='R² medio (test)')
fig.show()